# 03 — Model Evaluation

Evaluate discrimination, calibration and error rates on held-out synthetic examples.

**Data:** Synthetic invoice and purchase-order records. No production data. Not a bank credit scoring model.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root / "src"))

In [ ]:
from proofflow.ml import synthetic_invoices, split_data, build_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
x_train,x_test,y_train,y_test = split_data(synthetic_invoices())
model = build_pipeline().fit(x_train,y_train)
scores = model.predict_proba(x_test)[:,1]
print("ROC-AUC:",round(roc_auc_score(y_test,scores),3))
print("Average precision:",round(average_precision_score(y_test,scores),3))
print("Brier score:",round(brier_score_loss(y_test,scores),3))
print("Confusion matrix @0.5:\n",confusion_matrix(y_test,scores>=0.5))
print(classification_report(y_test,scores>=0.5,zero_division=0))

In [ ]:
fraction, mean_prediction=calibration_curve(y_test,scores,n_bins=8,strategy="quantile")
plt.plot([0,1],[0,1],"--",label="perfect calibration")
plt.plot(mean_prediction,fraction,"o-",label="model")
plt.xlabel("Mean predicted probability");plt.ylabel("Fraction needs review");plt.legend();plt.show()

**Interpretation:** The Brier score and calibration plot help identify misleading probability estimates. Do not treat a 0.5 threshold as a validated operational policy.